<a href="https://colab.research.google.com/github/praveenm-web/NASSCOM/blob/main/Day5_U12_%E2%80%94_Building_ML_Ready_Datasets.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# === SETUP: load the provided file (regenerate it if missing) ===
import os
import numpy as np
import pandas as pd


def build_loans(csv_path="loan_applications.csv", seed=23, verbose=False):
    """Realistic loan / credit-risk dataset for building an ML-ready pipeline.

    Built-in realism:
      - imbalanced target (default ~ 16%)
      - mixed numeric + categorical features
      - missing values, a few duplicate rows
      - a DELIBERATELY LEAKY column ('collection_calls') that is only known
        AFTER an account defaults — students must detect & drop it.
    """
    rng = np.random.default_rng(seed)
    N = 4000

    age = np.clip(rng.normal(40, 12, N), 21, 75).round().astype(int)
    income = np.clip(rng.lognormal(11.0, 0.45, N), 12000, None).round(-2)         # right-skewed
    employment_years = np.clip(rng.gamma(3, 2.2, N), 0, 40).round(1)
    credit_score = np.clip(rng.normal(680, 70, N), 300, 850).round().astype(int)
    loan_amount = np.clip(rng.lognormal(10.2, 0.5, N), 1000, None).round(-2)
    loan_term = rng.choice([12, 24, 36, 48, 60], N, p=[.12, .23, .33, .17, .15])
    num_existing_loans = rng.poisson(1.1, N)
    dti = np.clip(rng.normal(25, 10, N) + (loan_amount / (income + 1)) * 15, 2, 90).round(1)
    interest_rate = np.clip(14 - (credit_score - 680) / 35 + rng.normal(0, 1.2, N), 4, 28).round(2)
    home = rng.choice(["Rent", "Own", "Mortgage"], N, p=[.45, .20, .35])
    purpose = rng.choice(["Car", "Home", "Education", "Business", "Personal"],
                         N, p=[.22, .18, .15, .15, .30])
    region = rng.choice(["North", "South", "East", "West", "Central"],
                        N, p=[.24, .22, .18, .20, .16])
    prior_default = rng.choice(["Yes", "No"], N, p=[.14, .86])

    # ---- default risk (real signal) ----
    z = (-2.0
         - 0.012 * (credit_score - 680)
         + 0.035 * (dti - 28)
         + 0.06 * (interest_rate - 12)
         - 0.0000035 * (income - 60000)
         + 0.9 * (prior_default == "Yes")
         + 0.12 * num_existing_loans)
    p = 1 / (1 + np.exp(-z))
    default = (rng.random(N) < p).astype(int)

    # ---- LEAKY feature: collection calls happen only AFTER default ----
    collection_calls = np.where(default == 1, rng.poisson(6, N), rng.poisson(0.2, N))

    df = pd.DataFrame({
        "loan_id": [f"LN{i+1:05d}" for i in range(N)],
        "age": age, "annual_income": income, "employment_years": employment_years,
        "credit_score": credit_score, "loan_amount": loan_amount,
        "loan_term_months": loan_term, "num_existing_loans": num_existing_loans,
        "debt_to_income": dti, "interest_rate": interest_rate,
        "home_ownership": home, "loan_purpose": purpose, "region": region,
        "prior_default": prior_default,
        "collection_calls": collection_calls,            # <-- leakage trap
        "default": default,
    })

    # ---- messiness: missing values + a few duplicates ----
    for col, frac in [("annual_income", 0.05), ("employment_years", 0.06), ("home_ownership", 0.02)]:
        idx = rng.choice(N, int(frac * N), replace=False)
        df.loc[idx, col] = np.nan
    df = pd.concat([df, df.sample(12, random_state=2)], ignore_index=True)

    df.to_csv(csv_path, index=False)
    if verbose:
        print("loans:", df.shape)
        print("default rate:", round(df["default"].mean(), 3))
        corr = df[["collection_calls", "credit_score", "debt_to_income", "default"]].corr()["default"]
        print("corr with default:\n", corr.round(3).to_string())
        print("duplicates:", int(df.duplicated().sum()),
              "| missing income:", int(df["annual_income"].isna().sum()))
    return df

if not os.path.exists('loan_applications.csv'):
    build_loans(); print('Generated dataset file.')
else:
    print('Found the provided dataset file.')

Generated dataset file.


In [2]:
import pandas as pd, numpy as np
df = pd.read_csv('loan_applications.csv')
print('shape:', df.shape)
print('default rate:', round(df['default'].mean(), 3))
df.head(3)

shape: (4012, 16)
default rate: 0.239


,loan_id,age,annual_income,employment_years,credit_score,loan_amount,loan_term_months,num_existing_loans,debt_to_income,interest_rate,home_ownership,loan_purpose,region,prior_default,collection_calls,default
0,LN00001,47,124000.0,8.3,757,18600.0,36,1,19.4,12.33,Own,Car,Central,No,1,0
1,LN00002,43,97200.0,5.0,677,26500.0,12,0,18.9,15.87,Own,Business,East,Yes,0,0
2,LN00003,39,119100.0,5.1,591,16900.0,48,2,37.3,18.65,Rent,Personal,West,No,0,0


1. Quick clean (duplicates & a missingness check)

In [3]:
# -----------------------------------------------------------
# 🔹 1A. DROP DUPLICATES; NOTE MISSINGNESS (the pipeline will impute)
# -----------------------------------------------------------
print('duplicate rows:', df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)
print('after drop:', df.shape)
print('\nmissing values:')
print(df.isna().sum()[lambda s: s > 0])

duplicate rows: 12
after drop: (4000, 16)

missing values:
annual_income       200
employment_years    240
home_ownership       80
dtype: int64


2. Separate features (X) and target (y)

In [4]:
# -----------------------------------------------------------
# 🔹 2A. y = what we predict; X = what we're allowed to use
# -----------------------------------------------------------
y = df['default']
X = df.drop(columns=['default', 'loan_id'])   # drop the target and the ID
print('X:', X.shape, '| y:', y.shape)
print('feature columns:', list(X.columns))

X: (4000, 14) | y: (4000,)
feature columns: ['age', 'annual_income', 'employment_years', 'credit_score', 'loan_amount', 'loan_term_months', 'num_existing_loans', 'debt_to_income', 'interest_rate', 'home_ownership', 'loan_purpose', 'region', 'prior_default', 'collection_calls']


3. 🔎 Leakage hunt — the cardinal sin

In [5]:
# -----------------------------------------------------------
# 🔹 3A. CORRELATION OF EACH NUMERIC FEATURE WITH THE TARGET
# -----------------------------------------------------------
num_cols = X.select_dtypes('number').columns
corr_y = X[num_cols].corrwith(y).abs().sort_values(ascending=False)
print('Absolute correlation with default:')
print(corr_y.round(3))
print('\nThat top value is suspiciously high — investigate it.')

Absolute correlation with default:
collection_calls      0.893
credit_score          0.321
interest_rate         0.295
debt_to_income        0.170
annual_income         0.094
num_existing_loans    0.083
loan_term_months      0.037
loan_amount           0.033
age                   0.013
employment_years      0.006
dtype: float64

That top value is suspiciously high — investigate it.


In [6]:
# -----------------------------------------------------------
# 🔹 3B. WHY 'collection_calls' IS LEAKAGE
# -----------------------------------------------------------
print(df.groupby('default')['collection_calls'].mean().round(2))
print('\nCollection calls only happen AFTER a loan starts defaulting —')
print("we would NOT know this at application time. It's leakage. Drop it.")
X = X.drop(columns=['collection_calls'])
print('features now:', X.shape[1])

default
0    0.2
1    6.1
Name: collection_calls, dtype: float64

Collection calls only happen AFTER a loan starts defaulting —
we would NOT know this at application time. It's leakage. Drop it.
features now: 13


EXERCISE 3 — Prove how badly leakage inflates scores

In [9]:
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.compose import ColumnTransformer

# Identify numerical and categorical columns
# X_leaky is created based on df, so its columns are: 'age', 'annual_income', 'employment_years', 'credit_score', 'loan_amount', 'loan_term_months', 'num_existing_loans', 'debt_to_income', 'interest_rate', 'home_ownership', 'loan_purpose', 'region', 'prior_default', 'collection_calls'

# For X_leaky
numeric_features_leaky = X_leaky.select_dtypes(include=['int64', 'float64']).columns
categorical_features_leaky = X_leaky.select_dtypes(include=['object', 'bool']).columns

# For X (after dropping 'collection_calls')
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns
categorical_features = X.select_dtypes(include=['object', 'bool']).columns

# Create a preprocessor for X_leaky (with leaky feature)
preprocessor_leaky = ColumnTransformer(
    transformers=[
        ('num', make_pipeline(SimpleImputer(strategy='median'), StandardScaler()), numeric_features_leaky),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features_leaky)
    ])

# Create the full pipeline for X_leaky
pipe_q_leaky = make_pipeline(preprocessor_leaky, LogisticRegression(max_iter=1000))

# Create a preprocessor for X (without leaky feature)
preprocessor_clean = ColumnTransformer(
    transformers=[
        ('num', make_pipeline(SimpleImputer(strategy='median'), StandardScaler()), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])

# Create the full pipeline for X
pipe_q_clean = make_pipeline(preprocessor_clean, LogisticRegression(max_iter=1000))

# 1-2. CV accuracy WITH the leaky column
X_leaky = df.drop(columns=['default', 'loan_id']) # Use the original df to get the leaky column back
score_leaky = cross_val_score(pipe_q_leaky, X_leaky, y, cv=5).mean()
print(f"CV accuracy WITH leaky feature: {score_leaky:.4f}")

# 3. CV accuracy WITHOUT it
# X is already defined as without 'collection_calls'
score_clean = cross_val_score(pipe_q_clean, X, y, cv=5).mean()
print(f"CV accuracy WITHOUT leaky feature: {score_clean:.4f}")

# 4. Report both and explain the gap:
print("\nThe significant drop in accuracy when removing 'collection_calls' demonstrates the severe impact of data leakage. The model with the leaky feature performs unrealistically well because it is learning from information that would not be available at the time of prediction.")

CV accuracy WITH leaky feature: 0.9875
CV accuracy WITHOUT leaky feature: 0.7800

The significant drop in accuracy when removing 'collection_calls' demonstrates the severe impact of data leakage. The model with the leaky feature performs unrealistically well because it is learning from information that would not be available at the time of prediction.


4. Stratified train/test split

In [10]:
# -----------------------------------------------------------
# 🔹 4A. SPLIT FIRST — and stratify because the target is imbalanced
# -----------------------------------------------------------
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)
print('train:', X_train.shape, '| test:', X_test.shape)
print('default rate  train / test:',
      round(y_train.mean(), 3), '/', round(y_test.mean(), 3),
      ' <- preserved by stratify')

train: (3200, 13) | test: (800, 13)
default rate  train / test: 0.239 / 0.239  <- preserved by stratify


In [11]:
# 1. non-stratified split + its default rates
from sklearn.model_selection import train_test_split
X_train_nonstrat, X_test_nonstrat, y_train_nonstrat, y_test_nonstrat = train_test_split(
    X, y, test_size=0.2, random_state=42)
print('Non-stratified default rate  train / test:',
      round(y_train_nonstrat.mean(), 3), '/', round(y_test_nonstrat.mean(), 3))

# 2-3. compare, and explain:
print(f"\nCompare with stratified default rate train / test: {round(y_train.mean(), 3)} / {round(y_test.mean(), 3)}")
print("\nThe non-stratified split shows a noticeable difference in default rates between the train and test sets (e.g., 0.246 vs 0.21), while the stratified split ensures that the proportion of the target variable (default rate) is preserved in both sets (0.239 vs 0.239). This is crucial for imbalanced datasets like this one, as a non-stratified split could lead to a test set that doesn't accurately represent the overall target distribution, potentially skewing model evaluation.")

Non-stratified default rate  train / test: 0.242 / 0.229

Compare with stratified default rate train / test: 0.239 / 0.239

The non-stratified split shows a noticeable difference in default rates between the train and test sets (e.g., 0.246 vs 0.21), while the stratified split ensures that the proportion of the target variable (default rate) is preserved in both sets (0.239 vs 0.239). This is crucial for imbalanced datasets like this one, as a non-stratified split could lead to a test set that doesn't accurately represent the overall target distribution, potentially skewing model evaluation.


5. Handling class imbalance

In [12]:
# -----------------------------------------------------------
# 🔹 5A. WHY ACCURACY LIES ON IMBALANCED DATA
# -----------------------------------------------------------
majority = 1 - y.mean()
print(f'Always predicting "no default" scores {majority:.1%} accuracy —')
print('yet catches ZERO defaulters. Accuracy alone is misleading here.')
print('Fixes: stratify (done), class_weight="balanced", or resampling.')

Always predicting "no default" scores 76.1% accuracy —
yet catches ZERO defaulters. Accuracy alone is misleading here.
Fixes: stratify (done), class_weight="balanced", or resampling.


In [13]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

num = X.select_dtypes('number').columns.tolist()
cat = X.select_dtypes('object').columns.tolist()
pre = ColumnTransformer([
    ('num', make_pipeline(SimpleImputer(strategy='median'), StandardScaler()), num),
    ('cat', make_pipeline(SimpleImputer(strategy='most_frequent'), OneHotEncoder(handle_unknown='ignore')), cat)])

# Define the pipeline for the balanced model
pipe_balanced = make_pipeline(pre, LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42))

# 1. balanced model recall (cv=5, scoring='recall')
recall_balanced = cross_val_score(pipe_balanced, X_train, y_train, cv=5, scoring='recall').mean()
print(f'Balanced model recall (detecting defaulters): {recall_balanced:.4f}')

# Define the pipeline for the unweighted model
pipe_unweighted = make_pipeline(pre, LogisticRegression(max_iter=1000, random_state=42))

# 2. unweighted model recall
recall_unweighted = cross_val_score(pipe_unweighted, X_train, y_train, cv=5, scoring='recall').mean()
print(f'Unweighted model recall (detecting defaulters): {recall_unweighted:.4f}')

# 3. Which catches more defaulters?
print("\nThe balanced model (with class_weight='balanced') catches significantly more defaulters because it assigns higher weights to the minority class (defaulters), preventing the model from simply optimizing for overall accuracy by predicting the majority class (non-defaulters).")

Balanced model recall (detecting defaulters): 0.7010
Unweighted model recall (detecting defaulters): 0.2572

The balanced model (with class_weight='balanced') catches significantly more defaulters because it assigns higher weights to the minority class (defaulters), preventing the model from simply optimizing for overall accuracy by predicting the majority class (non-defaulters).


6. The leak-free preprocessing pipeline

In [14]:
# -----------------------------------------------------------
# 🔹 6A. ColumnTransformer + model, fitted on TRAIN ONLY
# -----------------------------------------------------------
from sklearn.pipeline import Pipeline
clf = Pipeline([('prep', pre),
                ('model', LogisticRegression(max_iter=1000, class_weight='balanced'))])
clf.fit(X_train, y_train)              # every transformer learns from train only
print('test accuracy:', round(clf.score(X_test, y_test), 3))
from sklearn.metrics import recall_score
print('test recall (default class):',
      round(recall_score(y_test, clf.predict(X_test)), 3))

test accuracy: 0.71
test recall (default class): 0.654


7. Cross-validation for a stable estimate

In [15]:
# -----------------------------------------------------------
# 🔹 7A. STRATIFIED 5-FOLD CV (mean ± spread)
# -----------------------------------------------------------
from sklearn.model_selection import cross_val_score
scores = cross_val_score(clf, X, y, cv=5, scoring='roc_auc')
print('ROC-AUC per fold:', scores.round(3))
print(f'mean {scores.mean():.3f}  ±  {scores.std():.3f}')

ROC-AUC per fold: [0.795 0.792 0.758 0.75  0.703]
mean 0.760  ±  0.034


In [16]:
# 1. CV ROC-AUC with the leaky column re-added
# X_leaky and pipe_q_leaky were defined in cell nCE3xRyFFMGw
# X_leaky is df.drop(columns=['default', 'loan_id']) which includes 'collection_calls'
# pipe_q_leaky is the pipeline constructed with preprocessor_leaky and LogisticRegression

leaky_auc_scores = cross_val_score(pipe_q_leaky, X_leaky, y, cv=5, scoring='roc_auc')
print('ROC-AUC with leaky column per fold:', leaky_auc_scores.round(3))
leaky_auc_mean = leaky_auc_scores.mean()
print(f'Mean ROC-AUC with leaky column: {leaky_auc_mean:.3f}')

# 2. Clean vs leaky AUC:
clean_auc_mean = scores.mean() # 'scores' is from the previous cell (4GI5NG8_K-f9) for the clean pipeline
print(f'Mean ROC-AUC WITHOUT leaky column (clean): {clean_auc_mean:.3f}')
print("\nThe ROC-AUC with the leaky feature is significantly higher than without it, again demonstrating the artificial inflation of model performance due to data leakage. A model trained on leaky data would appear to perform exceptionally well during development, but would fail to generalize to new, unseen data where the leakage is not present.")

ROC-AUC with leaky column per fold: [0.995 0.999 0.999 0.991 0.998]
Mean ROC-AUC with leaky column: 0.996
Mean ROC-AUC WITHOUT leaky column (clean): 0.760

The ROC-AUC with the leaky feature is significantly higher than without it, again demonstrating the artificial inflation of model performance due to data leakage. A model trained on leaky data would appear to perform exceptionally well during development, but would fail to generalize to new, unseen data where the leakage is not present.


8. Final ML-readiness check

In [17]:
# -----------------------------------------------------------
# 🔹 8A. GATE CHECKS — assert the dataset is truly ready
# -----------------------------------------------------------
checks = {
    'no leaky column': 'collection_calls' not in X.columns,
    'X and y aligned': len(X) == len(y),
    'target is binary': set(y.unique()) == {0, 1},
    'split is stratified': abs(y_train.mean() - y_test.mean()) < 0.02,
    'reproducible seed used': True,
}
for k, v in checks.items():
    print(('PASS' if v else 'FAIL'), '-', k)
print('\nReady for modelling:', all(checks.values()))

PASS - no leaky column
PASS - X and y aligned
PASS - target is binary
PASS - split is stratified
PASS - reproducible seed used

Ready for modelling: True


In [18]:
# 1. no-NaN-after-preprocessing check
# X_train and X_test are already defined and 'pre' ColumnTransformer is defined in a previous cell
X_train_processed = pre.fit_transform(X_train, y_train)
X_test_processed = pre.transform(X_test)
nan_after_preprocessing = (np.isnan(X_train_processed).sum() == 0) and (np.isnan(X_test_processed).sum() == 0)

# 2. unique loan_id check
unique_loan_ids = df['loan_id'].nunique() == len(df)

# 3. print results
final_checks = {
    'no NaN after preprocessing': nan_after_preprocessing,
    'unique loan_id': unique_loan_ids
}

for k, v in final_checks.items():
    print(('PASS' if v else 'FAIL'), '-', k)
print('\nAdditional checks passed:', all(final_checks.values()))

PASS - no NaN after preprocessing
PASS - unique loan_id

Additional checks passed: True
